## pyatspm Intersection Setup & Ingestion Notebook
**Current main branch reality (from repo inspection):**
- Recent commit moved workflow toward CLI ("Moved scripts workflow to cli"), but **scripts/** folder and files still exist in the structure.
- No root or src/atspm/cli.py is visible/confirmed as the entry point yet.
- README still documents **python scripts/setup_intersection.py** and **python scripts/run_ingestion.py** as the commands.
- Ingestion: default = incremental (append new .datZ via ingestion_log); **--full** = complete re-ingestion + cycle recompute.
- No --mode append/full/fill_gaps flags are documented or visible.
- README is lagging behind commit messages.

Assumes notebook is in **notebooks/** folder. Install package or run scripts directly.
Place .datZ files in **../intersections/<folder>/raw_data/** before ingesting.

## 1. Setup a New Intersection

In [ ]:
# Parameters
intersection_id   = 2068
intersection_name = "US-95 and SH-8"      # spaces become underscores in folder name
timezone          = "US/Mountain"

In [ ]:
!python ../scripts/setup_intersection.py --id {intersection_id} --name "{intersection_name}" --tz "{timezone}"

After running: copy your .datZ files into  \n
`../intersections/{intersection_id}_{name_with_underscores}/raw_data/`

## 2. Ingest a Single Intersection

In [ ]:
# Parameters
target_folder  = "2068_US-95_and_SH-8"   # folder name under intersections/
full_reprocess = False                   # True = --full (re-ingest ALL + recompute cycles)

In [ ]:
import subprocess

cmd = ["python", "../scripts/run_ingestion.py", "--target", target_folder]
if full_reprocess:
    cmd.append("--full")

print("Executing:", " ".join(cmd))
subprocess.run(cmd, check=True)

## 3. Batch Ingest Intersections

### Option A: By Explicit List

In [ ]:
# Parameters
target_folders = [
    "2068_US-95_and_SH-8",
    # "9999_Example_Rd_and_Main_St",
]

full_reprocess = False

In [ ]:
import subprocess

for target in target_folders:
    print(f"\n=== Ingesting {target} ===")
    cmd = ["python", "../scripts/run_ingestion.py", "--target", target]
    if full_reprocess:
        cmd.append("--full")
    subprocess.run(cmd, check=True)
    print(f"Done: {target}")

### Option B: All Folders in intersections/

In [ ]:
# Parameters
full_reprocess = False

In [ ]:
import os
import subprocess

intersections_dir = "../intersections"
folders = [
    d for d in os.listdir(intersections_dir)
    if os.path.isdir(os.path.join(intersections_dir, d))
    and not d.startswith(('.', '_'))
]

print(f"Found {len(folders)} intersection folders.")

for target in folders:
    print(f"\n=== Ingesting {target} ===")
    cmd = ["python", "../scripts/run_ingestion.py", "--target", target]
    if full_reprocess:
        cmd.append("--full")
    subprocess.run(cmd, check=True)
    print(f"Done: {target}")